# 08.9 · Exercises

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/08-deep-learning/09-augmentation-transfer-learning/exercises.ipynb)

In [ ]:
import os, warnings
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"        # hide TensorFlow's C++ start-up messages
warnings.filterwarnings("ignore")                # hide library deprecation notices
import numpy as np
import matplotlib.pyplot as plt
import keras
from keras import layers
keras.utils.set_random_seed(42)                  # same start every run (results still vary slightly by machine)
print("Keras", keras.__version__, "· backend", keras.backend.backend())

import pathlib

def flower_folder():
    """Download the 5-class flower photos once (229 MB) and return the folder."""
    path = keras.utils.get_file(
        "flower_photos", untar=True,
        origin="https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz")
    return next(p.parent for p in pathlib.Path(path).rglob("daisy") if p.is_dir())

def load_flowers(n_train=200, n_val=40, n_test=100, size=160, seed=0):
    """Pick n images per class for train/val/test; return uint8 arrays (small in memory)."""
    root = flower_folder()
    classes = sorted(p.name for p in root.iterdir() if p.is_dir())
    rng = np.random.default_rng(seed)
    parts = {"train": (0, n_train), "val": (n_train, n_train + n_val),
             "test": (n_train + n_val, n_train + n_val + n_test)}
    data = {k: ([], []) for k in parts}
    for label, name in enumerate(classes):
        files = sorted((root / name).glob("*.jpg"))
        rng.shuffle(files)
        for k, (a, b) in parts.items():
            for f in files[a:b]:
                img = keras.utils.load_img(f, target_size=(size, size))
                data[k][0].append(np.asarray(img, dtype="uint8"))
                data[k][1].append(label)
    out = {k: (np.stack(v[0]), np.array(v[1])) for k, v in data.items()}
    return out, classes

data, CLASSES = load_flowers(n_train=200, n_val=40, n_test=100)
(x_train, y_train), (x_val, y_val), (x_test, y_test) = data["train"], data["val"], data["test"]
print("classes:", CLASSES)
print("train", x_train.shape, " val", x_val.shape, " test", x_test.shape,
      f" ({x_train.nbytes / 1e6:.0f} MB as uint8)")

import time

augment = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),          # up to ±10% of a full turn (±36°)
    layers.RandomZoom(0.1),
], name="augment")

def fit_eval(model, epochs, lr=1e-3):
    """Train on the 1,000 photos, return (test accuracy, seconds)."""
    model.compile(optimizer=keras.optimizers.Adam(lr),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    start = time.time()
    model.fit(x_train, y_train, validation_data=(x_val, y_val),
              epochs=epochs, batch_size=32, verbose=0)
    return model.evaluate(x_test, y_test, verbose=0)[1], time.time() - start

def small_cnn(with_augmentation):
    steps = [keras.Input((160, 160, 3)), layers.Rescaling(1 / 255)]
    steps += [augment] if with_augmentation else []
    steps += [layers.Conv2D(16, 3, activation="relu"), layers.MaxPooling2D(),
              layers.Conv2D(32, 3, activation="relu"), layers.MaxPooling2D(),
              layers.Conv2D(64, 3, activation="relu"), layers.MaxPooling2D(),
              layers.GlobalAveragePooling2D(), layers.Dense(5, activation="softmax")]
    return keras.Sequential(steps)

def transfer_model(base_fn, n_classes=5):
    """Pretrained base (frozen) + augmentation + a new 5-class head."""
    base = base_fn(input_shape=(160, 160, 3), include_top=False, weights="imagenet", pooling="avg")
    base.trainable = False
    inputs = keras.Input((160, 160, 3))
    x = base(augment(inputs), training=False)
    outputs = layers.Dense(n_classes, activation="softmax")(layers.Dropout(0.2)(x))
    return keras.Model(inputs, outputs), base

def trainable(model):
    return sum(int(np.prod(w.shape)) for w in model.trainable_weights)

## Exercise 1: count the head by hand

EfficientNetB0 produces 1,280 features per photo. How many trainable weights does a new 5-class `Dense` head have? Build the model with `transfer_model(keras.applications.EfficientNetB0)` and check.

In [ ]:
# your code here

## Exercise 2: a bigger pretrained base

Train the EfficientNetB0 transfer model (frozen, 10 epochs). Compare test accuracy and training time with MobileNetV3Small. Which would you put on ShopWave's phone app, and which on a server?

In [ ]:
# your code here

## Exercise 3: how many photos do you need?

With the frozen MobileNetV3Small model, train on 25, 50, 100 and 200 photos per class (use the first *n* of each class in `x_train`) and plot test accuracy against *n*.

In [ ]:
# your code here

## Exercise 4: does augmentation still help?

Train the frozen MobileNetV3Small model without the augmentation layer. Compare with the lesson's result.

In [ ]:
# your code here